# Comparison of Statistical POS Taggers – Trigrams’n Tags (TnT)


## Introduction

This notebook is part of a series of three experiments comparing different **statistical part-of-speech taggers** using the **Brown Corpus**.  
Each notebook focuses on a specific tagging approach; this one evaluates the **Trigrams’n Tags (TnT)**, as implemented in NLTK.

The workflow includes:
- training the tagger on a subset of the Brown corpus,
- predicting POS tags on unseen data,
- evaluating performance using standard classification metrics,
- exporting sample results for inspection.

## References on the TnT Tagger
- Brants, Thorsten. 2000. TnT – A Statistical Part-ofSpeech Tagger. In Proceedings of the Sixth Applied Natural Language Processing Conference (ANLP 2000), Seattle, WA, pp. 224–231  
  


## Imports and Dependencies

We import standard Python libraries for timing, file export, and evaluation,
as well as NLTK modules required for corpus access, tagging, and tagset mapping.

In [1]:
import nltk

# Ensure required NLTK resources are available
resources = [
    ("corpora/brown", "brown"),
    ("taggers/averaged_perceptron_tagger", "averaged_perceptron_tagger"),
]

for path, name in resources:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(name)

In [2]:
import csv
import time
import datetime

from sklearn import metrics

from nltk.tag.tnt import TnT

from nltk.corpus import brown as corpus
corpus_name = "Brown"


## Dataset Split and Experimental Setup

We generate training and test partitions from the Brown corpus.
The first **105 files** are used for training and the subsequent **50 files** for testing.

Files are taken in their original order, and **no shuffling is applied**.

This fixed split is used to ensure that **all taggers in the comparison are trained and evaluated on exactly the same data**.


In [3]:
# ======================
# DATASET SPLIT
# ======================

# Total number of files in the Brown corpus
corpus_length = len(corpus.fileids())
print(f"Total corpus files: {corpus_length}")

# Define training and testing size
training_length = 105
print(f'Training length: {training_length} files')
testing_length = 50
print(f'Testing length: {testing_length} files')

# Training set
# Load tagged sentences from the selected training files
# Each sentence is a list of (word, tag) tuples
training_sentences = corpus.tagged_sents(
    corpus.fileids()[0:training_length]
)
# Testing set
# Next 50 files of the Brown corpus
testing_sentences = corpus.tagged_sents(corpus.fileids()[105:156])

Total corpus files: 500
Training length: 105 files
Testing length: 50 files


## Training

Training is performed **from scratch**, meaning that no pre-trained weights are used. 

Training is performed using the **Trigrams’n Tags (TnT)** statistical tagger.
TnT is a **Hidden Markov Model (HMM)**–based approach that estimates transition
and emission probabilities from the training corpus.

The total training time is measured to provide an indication of the computational cost of this approach on standard hardware.Training time is significantly shorter than for discriminative models such as 
the Averaged Perceptron.

In [4]:
# ======================
# TRAINING PHASE
# ======================
# This section trains a TnT POS tagger using
# a subset of the Brown Corpus.

# Initialize the TnT tagger
tagger = TnT()
print(f'Tagger initialized: {tagger}')

# Measure training time
start = time.time()

# Train the tagger on the training sentences
tagger.train(training_sentences)

end = time.time()

# Compute total training time
processing_time = end - start

print(f'Training performed in {processing_time:.2f} seconds')

Tagger initialized: <nltk.tag.tnt.TnT object at 0x00000272E26F4590>
Training performed in 2.17 seconds


## Brief Qualitative Evaluation on Unseen Data

This section provides a **qualitative inspection** of the tagger’s behavior on **unseen data**.  

The first five words of first two sentences from the testing set are selected to allow a direct comparison between the **gold-standard POS annotations** and the **POS tags predicted** by the trained tagger.

In [5]:
# BRIEF QUALITATIVE ANALYSIS
# Inspection of gold vs. predicted POS tags on a single unseen test document

# Select one test file (the first file after the training split)
test_file_id = corpus.fileids()[training_length]

# Retrieve gold-standard tagged sentences from the test file
qual_testing_sentences  = corpus.tagged_sents(test_file_id)

print(f"Qualitative evaluation on test file: {test_file_id}")
print("\nGold-standard labels (first tokens of first sentences):")
print(qual_testing_sentences[0][:5], qual_testing_sentences [1][:5])

# Remove POS tags to obtain raw word sequences
qual_bare_sentences = [[word for word, _ in sentence] for sentence in qual_testing_sentences ]
# Alternative:
# qual_bare_sentences = corpus.sents(test_file_id)

print("\nRaw words:")
print(qual_bare_sentences[0][:5], qual_bare_sentences[1][:5])

# Predict POS tags using the trained TnT tagger
qual_predicted_sentences = [tagger.tag([word for word,_ in sentence]) for sentence in qual_testing_sentences ]

print("\nPredicted labels (tagger default's tagset):")
print(qual_predicted_sentences[0][:5], qual_predicted_sentences[1][:5])

    

Qualitative evaluation on test file: ce01

Gold-standard labels (first tokens of first sentences):
[('Too', 'QL'), ('often', 'RB'), ('a', 'AT'), ('beginning', 'VBG'), ('bodybuilder', 'NN')] [('whaddya', 'WDT+BER+PP'), ('gonna', 'VBG+TO'), ('do', 'DO'), ('with', 'IN'), ('all', 'ABN')]

Raw words:
['Too', 'often', 'a', 'beginning', 'bodybuilder'] ['whaddya', 'gonna', 'do', 'with', 'all']

Predicted labels (tagger default's tagset):
[('Too', 'QL'), ('often', 'RB'), ('a', 'AT'), ('beginning', 'NN'), ('bodybuilder', 'Unk')] [('whaddya', 'Unk'), ('gonna', 'Unk'), ('do', 'DO'), ('with', 'IN'), ('all', 'ABN')]


## Quantitative Evaluation

This section provides a **quantitative evaluation** of the Trigrams'n tagger.

The test set consists of the **50 files from the Brown corpus ** that were **not used during training**.  
All sentences from these files are evaluated jointly in order to compute **global performance metrics**.

Gold-standard POS tags and predicted tags are flattened into single lists, allowing the computation of:
- accuracy,
- precision,
- recall,
- and F1-score,

using weighted averages to account for class imbalance.

In [6]:
# QUANTITATIVE EVALUATION
# Global performance metrics on the test set (50 files from the Brown corpus)

# Measure tagging time
start = time.time()

# Predict POS tags on the whole test set
predicted_sentences = [tagger.tag([word for word,_ in sentence]) for sentence in testing_sentences]

end = time.time()

# Compute total training time
processing_time = end - start

print(" ")
print("Predicted labels (tagger default's tagset):")  
print(predicted_sentences[0][0:5], predicted_sentences[1][0:5])
print('Tagging has lasted: ', processing_time, 'sec' )

# Flatten gold-standard POS tags from all test sentences
golds = [tag for sentence in testing_sentences for _,tag in sentence]

# Flatten predicted POS tags
predicted_labels = [tag for sentence in predicted_sentences for _,tag in sentence]

print("Gold labels (sample):")
print(golds[:5])

print("\nPredicted labels (sample):")
print(predicted_labels[:5])

# Compute evaluation metrics
# Weighted averages account for tag frequency imbalance
accuracy = metrics.accuracy_score(golds, predicted_labels)
precision = metrics.precision_score(golds, predicted_labels, average="weighted", zero_division=0)
recall = metrics.recall_score(golds, predicted_labels, average="weighted", zero_division=0)
f1_score = metrics.f1_score(golds, predicted_labels, average="weighted", zero_division=0)
        
print("\nPerformance metrics for TnT Tagger:")
print(" Accuracy :", accuracy)
print(" Precision:", precision)
print(" Recall   :", recall)
print(" F1-Score :", f1_score)


 
Predicted labels (tagger default's tagset):
[('Too', 'QL'), ('often', 'RB'), ('a', 'AT'), ('beginning', 'NN'), ('bodybuilder', 'Unk')] [('whaddya', 'Unk'), ('gonna', 'Unk'), ('do', 'DO'), ('with', 'IN'), ('all', 'ABN')]
Tagging has lasted:  1341.8823919296265 sec
Gold labels (sample):
['QL', 'RB', 'AT', 'VBG', 'NN']

Predicted labels (sample):
['QL', 'RB', 'AT', 'NN', 'Unk']

Performance metrics for TnT Tagger:
 Accuracy : 0.8639891163912965
 Precision: 0.9410199350890361
 Recall   : 0.8639891163912965
 F1-Score : 0.896558095502901


## Exporting Sample Annotations for Qualitative Comparison

To support a **qualitative comparison among different POS taggers**, this section exports a small sample of annotated data from the test set.

The first **15 sentences** of the selected test document are extracted, and for each token the following information is stored:
- the word form,
- the gold-standard POS tag from the Brown corpus,
- the POS tag predicted by the TnT tagger.

Results are exported to a CSV file.

In [7]:
# ======================
# EXPORT FOR QUALITATIVE COMPARISON
# ======================
# Export gold vs. predicted POS tags for qualitative comparison 
# across different taggers.
#
# Note:
# Each row in the CSV corresponds to a single token.
# Sentence boundaries are implicit and can be reconstructed
# from the original sentence order.
    
# Timestamp for unique file naming
now = datetime.datetime.now().strftime("%y%m%d-%H%M")

tagger_name = 'TnT'
corpus_name = 'Brown'

# Output filename
output_filename = f"{now}-{tagger_name}-{corpus_name}-samples.csv"

# Number of sentences to export
n_sentences = 15

with open(output_filename, mode='w', newline='', encoding='utf-8') as dades:
    writer = csv.writer(dades, delimiter=',')

    # (Optional) header row for clarity
    writer.writerow(["word", "gold_tag", "predicted_tag"])

    # Iterate over the first n_sentences of the test set
    for i in range(min(n_sentences, len(testing_sentences))):
        for j in range(len(testing_sentences[i])):
            # Gold annotation: (word, gold_tag)
            word, gold_tag = testing_sentences[i][j]

            # Predicted annotation: (word, predicted_tag)
            _, predicted_tag = predicted_sentences[i][j]

            # Write combined row: word, gold_tag, predicted_tag
            writer.writerow((word, gold_tag, predicted_tag))

print("\nGold vs predicted labels (first token):")
print(
    testing_sentences[0][0][0],
    testing_sentences[0][0][1],
    predicted_sentences[0][0][1],
)
print(f"\nExported to: {output_filename}")



Gold vs predicted labels (first token):
Too QL QL

Exported to: 260224-1438-TnT-Brown-samples.csv
